# 模型测试

## 导入

In [1]:
import argparse
import numpy as np
import pandas as pd
import os
import random
import torch
import dgl
import pickle as pkl
from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, precision_recall_curve

## 变量定义

- KNN的训练样本数必须小于正常样本个数，KNN在训练时会取前`KNN_train_count`个正常样本做预训练
- neighbor_ratio表示KNN的邻居数比例，`K=KNN_train_count*neighbor_ratio`，KNN会用当前结点最近的K个结点判定该结点是否属于恶意样本
- `K=KNN_train_count*neighbor_ratio`必须大于1，计算结点间平均距离时`K-1`会作为分母


In [20]:
KNN_train_count = 12 # KNN训练样本数（训练样本必须小于正常样本个数，测试集中必须同时包含正常样本和异常样本，推荐使用60-80%的数据做训练）
repeat_count = 5 # knn重复实验次数（-1表示只运行一次）
neighbor_ratio = 0.5 # 邻居数比例，KNN训练样本数*neighbor_ratio（K*neighbor_ratio必须大于0，否则KNN无法训练）

# 模型参数需要与训练时一致
Time =  False
optimizer_name = "ADAM"
weight_decay = 5e-4
lr = lr = 0.005

dataset_name = "wget"

device = "cpu"

pooling = "mean"

time_dim = 8

num_hidden = 256
max_epoch = 5
num_layers = 4
negative_slope = 0.2 # leaky relu的负斜率
mask_rate = 0.5 # 掩码率
alpha_l = 3 # sce损失函数的幂次，a>1越大对错误样本惩罚越厉害，越小对错误样本越容忍（鲁棒性更高）

networkx_graph_dir = '../data/CICAPT_IIOT/networkx_graph/test-15-10/operation'
model_dir = '../models'
model_file = 'checkpoint-cicapt-operation-36-14-no-recon.pt'
result_dir = '../results'


## KNN模型

In [3]:
def evaluate_batch_level_using_knn(repeat, dataset, embeddings, labels):
    """
    使用KNN算法评估批量级别的分类性能
    
    参数:
        repeat: 重复实验次数,-1表示只运行一次
        dataset: 数据集名称
        embeddings: 特征嵌入向量
        labels: 对应标签(0表示正常样本,1表示攻击样本)
    
    返回:
        AUC均值(和标准差)或单次AUC值
    """
    x, y = embeddings, labels
    train_count = KNN_train_count  # KNN训练样本数(仅使用正常样本)
    n_neighbors = min(int(train_count * neighbor_ratio), 10)  # 邻居数,不超过10
    
    # 分离正常样本和攻击样本的索引
    benign_idx = np.where(y == 0)[0]  # 正常样本索引
    attack_idx = np.where(y == 1)[0]  # 攻击样本索引
    
    if repeat != -1:  # 多次重复实验模式（重复repeat次）
        # 初始化各指标列表
        prec_list = []  # 精确率
        rec_list = []   # 召回率
        f1_list = []    # F1分数
        tp_list = []    # 真正例
        fp_list = []    # 假正例
        tn_list = []    # 真负例
        fn_list = []    # 假负例
        auc_list = []   # AUC值
        
        for s in range(repeat):
            set_random_seed(s)  # 设置随机种子保证可重复性
            # 打乱样本顺序
            np.random.shuffle(benign_idx)
            np.random.shuffle(attack_idx)
            
            # 构建训练集(仅使用正常样本)和测试集
            x_train = x[benign_idx[:train_count]]
            x_test = np.concatenate([x[benign_idx[train_count:]], x[attack_idx]], axis=0)
            y_test = np.concatenate([y[benign_idx[train_count:]], y[attack_idx]], axis=0)
            
            # 标准化特征
            x_train_mean = x_train.mean(axis=0)
            x_train_std = x_train.std(axis=0)
            x_train = (x_train - x_train_mean) / (x_train_std + 1e-6)  # 防止除零
            x_test = (x_test - x_train_mean) / (x_train_std + 1e-6)
            
            # 训练KNN模型
            nbrs = NearestNeighbors(n_neighbors=n_neighbors)
            nbrs.fit(x_train)
            # 对于x_train，计算每个样本的邻居距离
            # distances: [len(x_train), n_neighbors], 每个样本的邻居距离
            # indexes: [len(x_train), n_neighbors], 每个样本的邻居索引
            distances, indexes = nbrs.kneighbors(x_train, n_neighbors=n_neighbors)
            mean_distance = distances.mean() * n_neighbors / (n_neighbors - 1)
            distances, indexes = nbrs.kneighbors(x_test, n_neighbors=n_neighbors)

            # 计算异常分数，越大越异常
            score = distances.mean(axis=1) / mean_distance

            # 计算指标
            auc = roc_auc_score(y_test, score)
            prec, rec, threshold = precision_recall_curve(y_test, score)
            f1 = 2 * prec * rec / (rec + prec + 1e-9)
            max_f1_idx = np.argmax(f1)
            best_thres = threshold[max_f1_idx]
            prec_list.append(prec[max_f1_idx])
            rec_list.append(rec[max_f1_idx])
            f1_list.append(f1[max_f1_idx])

            tn = 0
            fn = 0
            tp = 0
            fp = 0
            for i in range(len(y_test)):
                if y_test[i] == 1.0 and score[i] >= best_thres:
                    tp += 1
                if y_test[i] == 1.0 and score[i] < best_thres:
                    fn += 1
                if y_test[i] == 0.0 and score[i] < best_thres:
                    tn += 1
                if y_test[i] == 0.0 and score[i] >= best_thres:
                    fp += 1
            
            # 记录本轮结果
            prec_list.append(prec[max_f1_idx])
            rec_list.append(rec[max_f1_idx])
            f1_list.append(f1[max_f1_idx])
            tp_list.append(tp)
            fp_list.append(fp)
            fn_list.append(fn)
            tn_list.append(tn)
            auc_list.append(auc)
        
        # 输出多次实验的统计结果
        print('AUC: {}+{}'.format(np.mean(auc_list), np.std(auc_list)))
        print('F1: {}+{}'.format(np.mean(f1_list), np.std(f1_list)))
        print('PRECISION: {}+{}'.format(np.mean(prec_list), np.std(prec_list)))
        print('RECALL: {}+{}'.format(np.mean(rec_list), np.std(rec_list)))
        print('TN: {}+{}'.format(np.mean(tn_list), np.std(tn_list)))
        print('FN: {}+{}'.format(np.mean(fn_list), np.std(fn_list)))
        print('TP: {}+{}'.format(np.mean(tp_list), np.std(tp_list)))
        print('FP: {}+{}'.format(np.mean(fp_list), np.std(fp_list)))
        return np.mean(auc_list), np.std(auc_list)
    
    else:  # 单次实验模式
        set_random_seed(0)
        np.random.shuffle(benign_idx)
        np.random.shuffle(attack_idx)
        
        # 构建训练测试集(同上)
        x_train = x[benign_idx[:train_count]]
        x_test = np.concatenate([x[benign_idx[train_count:]], x[attack_idx]], axis=0)
        y_test = np.concatenate([y[benign_idx[train_count:]], y[attack_idx]], axis=0)
        
        # 标准化特征
        x_train_mean = x_train.mean(axis=0)
        x_train_std = x_train.std(axis=0)
        x_train = (x_train - x_train_mean) / x_train_std
        x_test = (x_test - x_train_mean) / x_train_std
        
        # 训练KNN模型并计算异常分数(同上)
        nbrs = NearestNeighbors(n_neighbors=n_neighbors)
        nbrs.fit(x_train)
        distances, indexes = nbrs.kneighbors(x_train, n_neighbors=n_neighbors)
        mean_distance = distances.mean() * n_neighbors / (n_neighbors - 1)
        distances, indexes = nbrs.kneighbors(x_test, n_neighbors=n_neighbors)

        score = distances.mean(axis=1) / mean_distance
        
        # 计算评估指标
        auc = roc_auc_score(y_test, score)
        prec, rec, threshold = precision_recall_curve(y_test, score)
        f1 = 2 * prec * rec / (rec + prec + 1e-9)
        best_idx = np.argmax(f1)
        best_thres = threshold[best_idx]

        tn = 0
        fn = 0
        tp = 0
        fp = 0
        for i in range(len(y_test)):
            if y_test[i] == 1.0 and score[i] >= best_thres:
                tp += 1
            if y_test[i] == 1.0 and score[i] < best_thres:
                fn += 1
            if y_test[i] == 0.0 and score[i] < best_thres:
                tn += 1
            if y_test[i] == 0.0 and score[i] >= best_thres:
                fp += 1
        
        # 输出单次实验结果
        print('AUC: {}'.format(auc))
        print('F1: {}'.format(f1[best_idx]))
        print('PRECISION: {}'.format(prec[best_idx]))
        print('RECALL: {}'.format(rec[best_idx]))
        print('TN: {}'.format(tn))
        print('FN: {}'.format(fn))
        print('TP: {}'.format(tp))
        print('FP: {}'.format(fp))
        return auc, 0.0

## 数据加载

#### 从networkx图(graphs.pkl)读取原始数据（用于测试的数据）

In [4]:
# 必须定义数据对象才能使用pkl进行反序列化
class WgetDataset(dgl.data.DGLDataset):
    def process(self):
        pass

    def __init__(self, name):
        super(WgetDataset, self).__init__(name=name)
        if name == "wget":
            path = final_data_dir

            num_graphs = sample_nums
            self.graphs = []
            self.labels = []
            print("Loading {} dataset...".format(name))
            for i in tqdm(range(num_graphs)):
                idx = i
                g = dgl.from_networkx(
                    nx.node_link_graph(
                        json.load(open("{}/{}.json".format(path, str(idx))))
                    ),
                    node_attrs=["type"],
                    edge_attrs=["type"],
                )
                self.graphs.append(g)
                if 0 <= idx < attack_nums:  # 恶意数据标为1
                    self.labels.append(1)
                else:  # 良性数据标为0
                    self.labels.append(0)
        else:
            raise NotImplementedError
    # 返回元组，第一个元素是dgl图，第二个元素是标签
    def __getitem__(self, i):
        return self.graphs[i], self.labels[i]

    def __len__(self):
        return len(self.graphs)

In [5]:
# 从pkl文件读取构建好的networkx图
def load_rawdata(name):
    path = networkx_graph_dir

    if os.path.exists(path + '/graphs.pkl'):
        print('Loading processed {} dataset...'.format(name))
        raw_data = pkl.load(open(path + '/graphs.pkl', 'rb'))
    else:
        raise FileNotFoundError(f"File {path}/graphs.pkl not found!")
    
    return raw_data

#### 提取特征维度，构建数据集

In [6]:
def load_batch_level_dataset(dataset_name):
    dataset = load_rawdata(dataset_name)
    graph, _ = dataset[0]
    node_feature_dim = 0
    for g, _ in dataset:
        node_feature_dim = max(node_feature_dim, g.ndata["type"].max().item()) # 结点特征维度：最大结点类型+1
    edge_feature_dim = 0
    for g, _ in dataset:
        edge_feature_dim = max(edge_feature_dim, g.edata["type"].max().item()) # 边特征维度：最大边类型+1
    node_feature_dim += 1
    edge_feature_dim += 1
    full_dataset = [i for i in range(len(dataset))]
    train_dataset = [i for i in range(len(dataset)) if dataset[i][1] == 0] # 训练数据全部是良性数据，学习正常模式
    print('[n_graph, n_node_feat, n_edge_feat]: [{}, {}, {}]'.format(len(dataset), node_feature_dim, edge_feature_dim))

    return {'dataset': dataset,
            'train_index': train_dataset,
            'full_index': full_dataset,
            'n_feat': node_feature_dim,
            'e_feat': edge_feature_dim}

## 测试准备

#### 设置随机种子

In [7]:
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.determinstic = True

#### 构建模型

##### 完全模型

In [ ]:
from model.my_model import MYModel

def build_model(args):
    num_hidden = args.num_hidden
    num_layers = args.num_layers
    negative_slope = args.negative_slope
    mask_rate = args.mask_rate
    alpha_l = args.alpha_l
    n_dim = args.n_dim
    e_dim = args.e_dim + args.t_dim

    model = MYModel(
        n_dim=n_dim,
        e_dim=e_dim,
        hidden_dim=num_hidden,
        n_layers=num_layers,
        n_heads=4,
        activation="prelu",
        feat_drop=0.1,
        negative_slope=negative_slope,
        residual=True,
        mask_rate=mask_rate,
        norm="BatchNorm",
        loss_fn="sce",
        alpha_l=alpha_l,
    )

    return model

##### 无结点掩码

In [ ]:
from model.model_no_mask import MYModel


def build_model_no_mask(args):
    num_hidden = args.num_hidden
    num_layers = args.num_layers
    negative_slope = args.negative_slope
    mask_rate = args.mask_rate
    alpha_l = args.alpha_l
    n_dim = args.n_dim
    e_dim = args.e_dim + args.t_dim

    model = MYModel(
        n_dim=n_dim,
        e_dim=e_dim,
        hidden_dim=num_hidden,
        n_layers=num_layers,
        n_heads=4,
        activation="prelu",
        feat_drop=0.1,
        negative_slope=negative_slope,
        residual=True,
        mask_rate=mask_rate,
        norm="BatchNorm",
        loss_fn="sce",
        alpha_l=alpha_l,
    )

    return model

##### 无边重构

In [8]:
from model.model_no_recon import MYModel

def build_model_no_recon(args):
    num_hidden = args.num_hidden
    num_layers = args.num_layers
    negative_slope = args.negative_slope
    mask_rate = args.mask_rate
    alpha_l = args.alpha_l
    n_dim = args.n_dim
    e_dim = args.e_dim + args.t_dim

    model = MYModel(
        n_dim=n_dim,
        e_dim=e_dim,
        hidden_dim=num_hidden,
        n_layers=num_layers,
        n_heads=4,
        activation="prelu",
        feat_drop=0.1,
        negative_slope=negative_slope,
        residual=True,
        mask_rate=mask_rate,
        norm="BatchNorm",
        loss_fn="sce",
        alpha_l=alpha_l,
    )

    return model

#### 定义测试过程

In [9]:
from model.utils import transform_graph


def batch_level_evaluation(
    model, pooler, device, method, dataset, res_path=None, n_dim=0, e_dim=0
):
    model.eval()  # 推理模式
    x_list = []
    y_list = []
    # KNN使用的数据集应该与模型训练使用的数据集区分开
    data = load_batch_level_dataset(dataset)
    full = data["full_index"]
    graphs = data["dataset"]
    with torch.no_grad():
        for i in full:
            g = transform_graph(graphs[i][0], n_dim, e_dim).to(device)  # g是图特征向量
            label = graphs[i][1]  # label是图的标签
            out = model.embed(g)  # 获取潜在空间的特征
            if dataset != "wget":
                out = (
                    pooler(g, out).cpu().numpy()
                )  # 将整张图池化为一个特征向量，作为特征摘要
            else:
                out = pooler(g, out, n_types=data["n_feat"]).cpu().numpy()
            y_list.append(label)
            x_list.append(out)
    x = np.concatenate(x_list, axis=0)
    y = np.array(y_list)

    if res_path is not None:
        os.makedirs(os.path.dirname(os.path.abspath(res_path)), exist_ok=True)
        num_feat = x.shape[1]
        col_names = [f"f{i}" for i in range(num_feat)] + ["label"]
        df = pd.DataFrame(np.hstack([x, y.reshape(-1, 1)]), columns=col_names)
        df.to_csv(res_path, index=False)

    if "knn" in method:
        test_auc, test_std = evaluate_batch_level_using_knn(repeat_count, dataset, x, y)
    else:
        raise NotImplementedError
    return test_auc, test_std

In [ ]:
from model.utils import transform_graph_with_time


def batch_level_evaluation_with_time(
    model, pooler, device, method, dataset, res_path=None, n_dim=0, e_dim=0, t_dim=0
):
    model.eval()  # 推理模式
    x_list = []
    y_list = []
    # KNN使用的数据集应该与模型训练使用的数据集区分开
    data = load_batch_level_dataset(dataset)
    full = data["full_index"]
    graphs = data["dataset"]
    with torch.no_grad():
        for i in full:
            g = transform_graph_with_time(graphs[i][0], n_dim, e_dim, t_dim).to(
                device
            )  # g是图特征向量
            label = graphs[i][1]  # label是图的标签
            out = model.embed(g)  # 获取潜在空间的特征
            if dataset != "wget":
                out = (
                    pooler(g, out).cpu().numpy()
                )  # 将整张图池化为一个特征向量，作为特征摘要
            else:
                out = pooler(g, out, n_types=data["n_feat"]).cpu().numpy()
            y_list.append(label)
            x_list.append(out)
    x = np.concatenate(x_list, axis=0)
    y = np.array(y_list)

    if res_path is not None:
        os.makedirs(os.path.dirname(os.path.abspath(res_path)), exist_ok=True)
        num_feat = x.shape[1]
        col_names = [f"f{i}" for i in range(num_feat)] + ["label"]
        df = pd.DataFrame(np.hstack([x, y.reshape(-1, 1)]), columns=col_names)
        df.to_csv(res_path, index=False)

    if "knn" in method:
        test_auc, test_std = evaluate_batch_level_using_knn(repeat_count, dataset, x, y)
    else:
        raise NotImplementedError
    return test_auc, test_std

## 执行处理

#### 构建参数

In [21]:
# 加载数据，提取结点和边的特征维数
dataset = load_batch_level_dataset(dataset_name)
n_dim = dataset["n_feat"]
e_dim = dataset["e_feat"]

if not Time:
    time_dim = 0

model_args = argparse.Namespace(
    num_hidden=num_hidden,
    num_layers=num_layers,
    negative_slope=negative_slope,
    mask_rate=mask_rate,
    alpha_l=alpha_l,
    n_dim=n_dim,
    e_dim=e_dim,
    t_dim=time_dim,
    pooling=pooling,
    dataset=dataset_name,
)

Loading processed wget dataset...
[n_graph, n_node_feat, n_edge_feat]: [25, 6, 12]


#### 加载模型并推理

In [22]:
from model.utils import Pooling

set_random_seed(0)

model = build_model_no_recon(model_args)

result_path = os.path.join(result_dir, model_file.replace('.pt', '.csv'))

model_path = os.path.join(model_dir, model_file)
# 加载模型参数
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
pooler = Pooling(model_args.pooling)
if Time:
    test_auc, test_std = batch_level_evaluation_with_time(
        model,
        pooler,
        device,
        ["knn"],
        model_args.dataset,
        result_path,
        model_args.n_dim,
        model_args.e_dim,
        model_args.t_dim,
    )
else:
    test_auc, test_std = batch_level_evaluation(
        model,
        pooler,
        device,
        ["knn"],
        model_args.dataset,
        result_path,
        model_args.n_dim,
        model_args.e_dim,
    )
print(f"#Test_AUC: {test_auc:.4f}±{test_std:.4f}")

d:\Desktop\UniAPT\scripts\model\model_no_recon.py:38: UserWarning: nn.init.xavier_uniform is now deprecated in favor of nn.init.xavier_uniform_.
  nn.init.xavier_uniform(m.weight)


Loading processed wget dataset...
[n_graph, n_node_feat, n_edge_feat]: [25, 6, 12]
AUC: 0.74+0.09285592184789415
F1: 0.8813350895379072+0.015767276711326723
PRECISION: 0.8282051282051283+0.08941331166237278
RECALL: 0.96+0.07999999999999999
TN: 0.8+1.1661903789690602
FN: 0.4+0.8
TP: 9.6+0.7999999999999999
FP: 2.2+1.16619037896906
#Test_AUC: 0.7400±0.0929
